In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist         # библиотека базы выборок Mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

x_train = np.expand_dims(x_train, axis=3)
x_test = np.expand_dims(x_test, axis=3)

print( x_train.shape )

model = keras.Sequential([
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2), strides=2),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    MaxPooling2D((2, 2), strides=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10,  activation='softmax')
])

# print(model.summary())      # вывод структуры НС в консоль

model.compile(optimizer='adam',
             loss='categorical_crossentropy',
             metrics=['accuracy'])


his = model.fit(x_train, y_train_cat, batch_size=32, epochs=5, validation_split=0.2)

model.evaluate(x_test, y_test_cat)

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, confusion_matrix
import seaborn as sns

# 1) Test məlumatları üzrə proqnoz (ehtimallar -> rəqəm)
pred_probs = model.predict(x_test)
pred = np.argmax(pred_probs, axis=1)

# 2) Hər rəqəm (0-9) üçün ətraflı hesabat
print(classification_report(y_test, pred, digits=4))

# 3) Ümumi (macro) precision və recall
precision = precision_score(y_test, pred, average='macro')
recall = recall_score(y_test, pred, average='macro')
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
# 4) Confusion matrix
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Proqnoz')
plt.ylabel('Əsl')
plt.title('Confusion Matrix')
plt.show()

## Izah

In [ ]:
import os
# os — Python-un əməliyyat sistemi ilə işləməsi üçün standart kitabxanadır.
# Burada TensorFlow-un terminalda çıxardığı bəzi texniki mesajları idarə etmək üçün istifadə olunur.


os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TF_CPP_MIN_LOG_LEVEL TensorFlow-un terminalda hansı səviyyədə mesaj
# göstərməsini müəyyən edən environment variable-dır.
#
# Burada istifadə olunan qiymətlər:
#
# '0' → bütün mesajları göstərir
# '1' → INFO mesajlarını gizlədir
# '2' → INFO və WARNING mesajlarını gizlədir
# '3' → INFO, WARNING və ERROR mesajlarını gizlədir
#
# Biz '2' seçmişik.
# Buna görə TensorFlow-un adi INFO və WARNING mesajları terminalda
# göstərilməyəcək.
#
# Bu sətrin TensorFlow import edilməzdən ƏVVƏL yazılması vacibdir.
# Çünki TensorFlow başladığı anda bu dəyişənin qiymətini oxuyur.


import numpy as np
# NumPy — Python-da ədədlər, massivlər və çoxölçülü array-lərlə
# işləmək üçün istifadə olunan əsas kitabxanalardan biridir.
#
# np — NumPy üçün verdiyimiz qısa addır.
#
# Bu proqramda NumPy-dan:
# - şəkillərə yeni ölçü (kanal ölçüsü) əlavə etmək üçün
#   expand_dims() funksiyasında istifadə edəcəyik.


import matplotlib.pyplot as plt
# matplotlib.pyplot — Python-da qrafik, şəkil və digər vizual
# elementləri göstərmək üçün istifadə olunan kitabxanadır.
#
# plt — matplotlib.pyplot üçün verdiyimiz qısa addır.
#
# Qeyd:
# Bu kodda plt import olunub, amma aşağıda heç bir yerdə istifadə olunmur.
# Yəni əvvəlki koddan fərqli olaraq burada şəkilləri ekranda göstərmirik.
# Lazım olarsa, sonradan şəkil və ya qrafik göstərmək üçün istifadə edilə bilər.


from tensorflow.keras.datasets import mnist         # библиотека базы выборок Mnist
# Yuxarıdakı rus dilindəki şərh belə tərcümə olunur:
# "MNIST nümunələr bazası kitabxanası".
#
# TensorFlow-un Keras hissəsində hazır MNIST dataset-i mövcuddur.
#
# MNIST — əl ilə yazılmış rəqəmlərdən ibarət məşhur datasetdir.
#
# Burada 0-dan 9-a qədər rəqəmlərin şəkilləri var.
#
# Hər bir şəkil 28 × 28 piksel ölçüsündədir.


from tensorflow import keras
# TensorFlow daxilindəki Keras API-ni import edirik.
#
# Keras neyron şəbəkəsi yaratmaq, compile etmək, öyrətmək,
# test etmək və proqnoz almaq üçün rahat interfeys təqdim edir.
#
# Aşağıda keras vasitəsilə:
# - model yaradacağıq (keras.Sequential)
# - to_categorical() istifadə edəcəyik.


from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D
# TensorFlow Keras-dan bu dəfə beş fərqli layer import edirik.
#
# Dense:
# → tam əlaqəli (fully connected) neyron şəbəkəsi qatıdır.
#
# Flatten:
# → çoxölçülü məlumatı birölçülü formaya çevirir.
#
# Dropout:
# → təlim zamanı neyronların müəyyən hissəsini təsadüfi şəkildə
#   "söndürən" qatdır. Məqsəd overfitting-in qarşısını almaqdır.
#   (Overfitting — modelin training məlumatlarını əzbərləməsi,
#   amma yeni məlumatlarda pis nəticə verməsi deməkdir.)
#
#   DİQQƏT: Dropout import olunub, amma aşağıdakı modeldə
#   istifadə olunmur. Yəni hələlik sadəcə "ehtiyat üçün" import edilib.
#
# Conv2D:
# → 2 ölçülü konvolyusiya qatıdır. Şəkillərdə kənarları, xətləri,
#   künc və digər xüsusiyyətləri tapmaq üçün istifadə olunur.
#
# MaxPooling2D:
# → şəkilin (və ya xüsusiyyət xəritəsinin) ölçüsünü kiçildən qatdır.
#   Hər kiçik bölgədən yalnız ən böyük qiyməti saxlayır.
#
# Əvvəlki kodda yalnız Dense və Flatten var idi.
# İndi Conv2D və MaxPooling2D əlavə olunduğu üçün
# bu şəbəkə artıq Convolutional Neural Network (CNN),
# yəni konvolyusiya neyron şəbəkəsi adlanır.


(x_train, y_train), (x_test, y_test) = mnist.load_data()
# mnist.load_data() hazır MNIST datasetini yükləyir.
#
# Dataset iki əsas hissəyə bölünür:
#
# 1. Training dataset
# 2. Test dataset
#
# x_train → training şəkilləridir.
# y_train → training şəkillərinin düzgün cavablarıdır.
#
# x_test → test şəkilləridir.
# y_test → test şəkillərinin düzgün cavablarıdır.
#
# Burada "x" giriş məlumatını,
# "y" isə düzgün cavabı ifadə edir.
#
# MNIST datasetində adətən:
#
# x_train → 60 000 şəkil  →  forması (60000, 28, 28)
# y_train → 60 000 cavab  →  forması (60000,)
#
# x_test → 10 000 şəkil   →  forması (10000, 28, 28)
# y_test → 10 000 cavab   →  forması (10000,)
#
# olur.


# стандартизация входных данных
# Yuxarıdakı rus dilindəki şərh: "giriş məlumatlarının standartlaşdırılması".
x_train = x_train / 255
# MNIST şəkillərində piksel qiymətləri 0 ilə 255 arasındadır.
#
# 0   → qara piksel
# 255 → ağ piksel
#
# Biz bütün qiymətləri 255-ə bölürük.
#
# Bunun nəticəsində:
#
# 0   / 255 = 0
# 128 / 255 ≈ 0.502
# 255 / 255 = 1
#
# Beləliklə bütün giriş məlumatlarımız 0 ilə 1 arasında olur.
#
# Bu əməliyyata normalization, yəni normallaşdırma deyilir.
#
# Neyron şəbəkələri üçün giriş məlumatlarının daha kiçik və
# eyni miqyasda olması təlim prosesini daha rahat və stabil edə bilər.


x_test = x_test / 255
# Eyni normallaşdırmanı test şəkillərinə də tətbiq edirik.
#
# Bu çox vacibdir.
#
# Əgər x_train-i 0–1 aralığına gətirib,
# x_test-i 0–255 aralığında saxlasaydıq,
# model training və test zamanı fərqli miqyasda məlumat görmüş olardı.
#
# Ona görə hər iki dataset eyni formada hazırlanır.


y_train_cat = keras.utils.to_categorical(y_train, 10)
# y_train-də cavablar adi rəqəmlər şəklindədir:
#
# 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
#
# Lakin modelin son qatında 10 neyron olacaq.
# Buna görə cavabları One-Hot Encoding formasına çeviririk.
#
# to_categorical() rəqəmləri kateqorik vektora çevirir.
#
# Buradakı 10:
# → ümumilikdə 10 kateqoriyamız olduğunu bildirir.
#
# Məsələn:
#
# 0 →
# [1,0,0,0,0,0,0,0,0,0]
#
# 1 →
# [0,1,0,0,0,0,0,0,0,0]
#
# 2 →
# [0,0,1,0,0,0,0,0,0,0]
#
# ...
#
# 9 →
# [0,0,0,0,0,0,0,0,0,1]
#
# Burada yalnız düzgün rəqəmə uyğun mövqedə 1,
# digər mövqelərdə isə 0 olur.
#
# Nəticədə y_train_cat-ın forması (60000, 10) olur.


y_test_cat = keras.utils.to_categorical(y_test, 10)
# Test cavablarını da One-Hot Encoding formasına çeviririk.
#
# Bunun səbəbi model.evaluate() zamanı test cavablarının
# modelin çıxışı ilə eyni formada olmasıdır.
#
# Beləliklə:
#
# y_train → y_train_cat
# y_test  → y_test_cat
#
# çevrilmiş olur.
# y_test_cat-ın forması (10000, 10) olur.


x_train = np.expand_dims(x_train, axis=3)
# BU SƏTİR ƏVVƏLKİ KODDAN ƏSAS FƏRQLƏRDƏN BİRİDİR.
#
# Conv2D qatı şəkilləri 4 ölçülü formada gözləyir:
#
# (şəkil sayı, hündürlük, en, kanal sayı)
#
# Amma x_train-in hazırkı forması:
#
# (60000, 28, 28)
#
# yəni yalnız 3 ölçü var. Kanal ölçüsü çatışmır.
#
# np.expand_dims(x_train, axis=3) array-ə yeni bir ölçü əlavə edir.
#
# axis=3 → yeni ölçü ən axırda (dördüncü mövqedə) əlavə olunur.
#
# Nəticədə:
#
# (60000, 28, 28)
#
# formasından:
#
# (60000, 28, 28, 1)
#
# formasına keçirik.
#
# Axırdakı 1:
# → kanal sayıdır.
# → 1 kanal = grayscale, yəni qara-ağ şəkil deməkdir.
#   (Rəngli RGB şəkil olsaydı, burada 3 olardı.)
#
# axis anlayışını belə düşün:
#
# axis=0 → birinci ölçü   (şəkil sayı)
# axis=1 → ikinci ölçü    (hündürlük)
# axis=2 → üçüncü ölçü    (en)
# axis=3 → dördüncü ölçü  (kanal)
#
# Əvvəlki kodda tək bir şəkil üçün expand_dims(..., axis=0) yazmışdıq,
# çünki orada əvvələ "batch" ölçüsü əlavə etmək lazım idi.
# Burada isə bütün dataset üçün ƏN SONA kanal ölçüsü əlavə edirik,
# ona görə axis=3 yazılır.


x_test = np.expand_dims(x_test, axis=3)
# Eyni əməliyyatı test şəkillərinə də tətbiq edirik.
#
# (10000, 28, 28)
#
# formasından:
#
# (10000, 28, 28, 1)
#
# formasına keçir.
#
# Training və test məlumatları eyni formada olmalıdır,
# yoxsa model test zamanı xəta verər.


print( x_train.shape )
# x_train massivinin formasını konsola çıxarır.
#
# Nəticə belə olacaq:
#
# (60000, 28, 28, 1)
#
# Burada:
#
# 60000 → training şəkillərinin sayı
# 28    → şəkilin hündürlüyü
# 28    → şəkilin eni
# 1     → kanal sayı (qara-ağ)
#
# Bu sətr expand_dims-in düzgün işlədiyini yoxlamaq üçündür.


model = keras.Sequential([
# Sequential model yaradırıq.
#
# Sequential o deməkdir ki, layer-lər ardıcıl şəkildə
# bir-birinin arxasınca işləyəcək.
#
# Bizim modelimizin quruluşu belə olacaq:
#
# Şəkil (28, 28, 1)
#   ↓
# Conv2D 32
#   ↓
# MaxPooling2D
#   ↓
# Conv2D 64
#   ↓
# MaxPooling2D
#   ↓
# Flatten
#   ↓
# Dense 128
#   ↓
# Dense 10
#
# Əvvəlki modeldə şəkil birbaşa Flatten-ə gedirdi.
# İndi isə əvvəlcə Conv2D və MaxPooling2D qatları
# şəkildən vacib xüsusiyyətləri çıxarır, sonra Dense qatları
# bu xüsusiyyətlərə əsasən qərar verir.


    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    # Birinci konvolyusiya qatı.
    #
    # Conv2D şəkil üzərində kiçik "pəncərə" (filter) gəzdirir
    # və hər mövqedə həmin bölgədəki piksellərə əsasən bir qiymət hesablayır.
    # Filterlər şəkildəki kənarları, xətləri, əyriləri və s. tapmağı öyrənir.
    #
    # 32:
    # → 32 ədəd fərqli filter istifadə olunur.
    # → Hər filter fərqli bir xüsusiyyət axtarır.
    # → Nəticədə 32 ədəd "xüsusiyyət xəritəsi" (feature map) alınır.
    #
    # (3,3):
    # → hər filterin ölçüsü 3 × 3 pikseldir.
    # → Yəni filter hər dəfə şəkilin 3 × 3 bölgəsinə baxır.
    #
    # padding='same':
    # → şəkilin ətrafına avtomatik olaraq sıfırlardan ibarət çərçivə əlavə edir.
    # → Bunun sayəsində çıxışın ölçüsü girişin ölçüsü ilə eyni qalır: 28 × 28.
    #
    #   Müqayisə üçün:
    #   padding='valid' (defolt) olsaydı, 3 × 3 filter kənarlara sığmadığı üçün
    #   çıxış 26 × 26 olardı.
    #
    # activation='relu':
    # → ReLU aktivasiya funksiyasıdır: f(x) = max(0, x)
    # → Mənfi qiymətlər 0 olur, müsbət qiymətlər olduğu kimi qalır.
    #
    # input_shape=(28, 28, 1):
    # → modelin ilk qatı olduğu üçün giriş formasını bildiririk.
    # → 28 hündürlük, 28 en, 1 kanal (qara-ağ).
    # → Bu, yuxarıda expand_dims ilə hazırladığımız formadır.
    #
    # Bu qatdan sonra çıxışın forması:
    #
    # (28, 28, 32)
    #
    # Parametr sayı:
    #
    # 3 × 3 × 1 × 32 + 32 = 320
    #
    # (3×3 çəkilər, 1 giriş kanalı, 32 filter və hər filter üçün 1 bias.)


    MaxPooling2D((2, 2), strides=2),
    # Birinci pooling qatı.
    #
    # MaxPooling xüsusiyyət xəritəsinin ölçüsünü kiçildir.
    #
    # (2, 2):
    # → 2 × 2 ölçülü pəncərə istifadə olunur.
    # → Hər 2 × 2 bölgədən yalnız ən böyük qiymət saxlanılır.
    #
    # Məsələn:
    #
    # [1, 3]
    # [2, 8]   →   8
    #
    # strides=2:
    # → pəncərə hər dəfə 2 piksel sürüşür.
    # → Pəncərələr bir-birinin üstünə düşmür.
    #
    # Nəticədə hər ölçü (hündürlük və en) 2 dəfə kiçilir:
    #
    # (28, 28, 32)  →  (14, 14, 32)
    #
    # Filter sayı (32) dəyişmir, yalnız ölçü kiçilir.
    #
    # Bu qatın öyrədilən parametri yoxdur (0 parametr).
    #
    # Niyə pooling istifadə edirik?
    # - Hesablamanı azaldır (məlumat 4 dəfə kiçilir)
    # - Ən vacib xüsusiyyətləri saxlayır
    # - Rəqəmin şəkildə bir az yerini dəyişməsinə qarşı
    #   modeli daha davamlı edir
    #
    # Qeyd:
    # Pooling-də strides verilməsə, defolt olaraq pəncərə ölçüsünə bərabər olur,
    # yəni burada strides=2 yazmasaq da eyni nəticə alınardı.


    Conv2D(64, (3,3), padding='same', activation='relu'),
    # İkinci konvolyusiya qatı.
    #
    # Bu dəfə 64 ədəd filter istifadə olunur.
    #
    # Burada input_shape yazmırıq,
    # çünki Keras giriş formasını əvvəlki qatdan özü bilir:
    # (14, 14, 32)
    #
    # Birinci Conv2D sadə xüsusiyyətləri (xətlər, kənarlar) tapırdı.
    # İkinci Conv2D isə əvvəlki xüsusiyyətləri birləşdirib
    # daha mürəkkəb formaları (əyrilər, döngələr, künclər) tapa bilir.
    #
    # padding='same' olduğu üçün ölçü dəyişmir:
    #
    # (14, 14, 32)  →  (14, 14, 64)
    #
    # Parametr sayı:
    #
    # 3 × 3 × 32 × 64 + 64 = 18 496
    #
    # (Hər filter 32 giriş kanalının hamısına baxır, ona görə 32-yə vurulur.)


    MaxPooling2D((2, 2), strides=2),
    # İkinci pooling qatı.
    #
    # Yenə ölçünü 2 dəfə kiçildir:
    #
    # (14, 14, 64)  →  (7, 7, 64)
    #
    # Yəni indi hər şəkil 7 × 7 ölçülü 64 ədəd
    # xüsusiyyət xəritəsi ilə təmsil olunur.


    Flatten(),
    # Flatten çoxölçülü məlumatı birölçülü formaya çevirir.
    #
    # Burada input_shape yazmağa ehtiyac yoxdur,
    # çünki forma əvvəlki qatdan avtomatik alınır.
    #
    # Formanın dəyişməsi:
    #
    # (7, 7, 64)  →  3136
    #
    # Çünki:
    #
    # 7 × 7 × 64 = 3136
    #
    # Dense qatları birölçülü giriş tələb etdiyi üçün
    # konvolyusiya qatlarından sonra Flatten istifadə edirik.
    #
    # Əvvəlki kodda Flatten ən əvvəldə idi (784 alınırdı),
    # indi isə Conv/Pooling qatlarından sonra gəlir (3136 alınır).


    Dense(128, activation='relu'),
    # 128 neyronlu Dense qatı yaradırıq.
    #
    # Dense tam əlaqəli, yəni fully connected layer-dir.
    #
    # Bu qatın hər bir neyronu əvvəlki 3136 giriş qiymətindən
    # məlumat alır.
    #
    # 128 → bu layer-də 128 neyron olduğunu bildirir.
    #
    # activation='relu'
    # → ReLU aktivasiya funksiyasından istifadə edilir.
    #
    # ReLU:
    #
    # f(x) = max(0,x)
    #
    # Bu qat Conv qatlarının çıxardığı xüsusiyyətləri birləşdirib
    # hansı rəqəm olduğuna qərar verməyə hazırlaşır.
    #
    # Parametr sayı:
    #
    # 3136 × 128 + 128 = 401 536
    #
    # Diqqət et: modelin parametrlərinin böyük hissəsi məhz bu qatdadır.


    Dense(10,  activation='softmax')
    # Bu modelin son Dense qatıdır.
    #
    # 10 → 10 neyron deməkdir.
    #
    # Niyə 10?
    #
    # Çünki MNIST-də 10 mümkün sinif var:
    #
    # 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
    #
    # Hər neyron bir rəqəmi təmsil edir.
    #
    # activation='softmax'
    # → çıxışları ehtimal formasına gətirir.
    #
    # Məsələn model bir şəkil üçün belə nəticə verə bilər:
    #
    # 0 → 0.01
    # 1 → 0.02
    # 2 → 0.01
    # 3 → 0.03
    # 4 → 0.01
    # 5 → 0.02
    # 6 → 0.01
    # 7 → 0.85
    # 8 → 0.01
    # 9 → 0.03
    #
    # Ən böyük ehtimal 7-yə aiddir.
    #
    # Deməli model:
    # "Bu şəkil 7-dir."
    #
    # nəticəsinə gəlir.
    #
    # Softmax nəticəsində bu ehtimalların cəmi təxminən 1 olur.
    #
    # Parametr sayı:
    #
    # 128 × 10 + 10 = 1 290
])
# Modelin ümumi parametr sayı:
#
# 320 + 0 + 18 496 + 0 + 0 + 401 536 + 1 290 = 421 642
#
# (Conv2D + Pool + Conv2D + Pool + Flatten + Dense + Dense)
#
# Bu rəqəmi model.summary() ilə də görə bilərsən.


# print(model.summary())      # вывод структуры НС в консоль
# Yuxarıdakı rus dilindəki şərh: "neyron şəbəkəsinin strukturunu konsola çıxarmaq".
#
# Bu sətr şərhə (comment) çevrilib, yəni işləmir.
# İşlətmək istəsən, əvvəlindəki # işarəsini silmək kifayətdir.
#
# model.summary() modelin strukturunu konsolda göstərir:
# - layer-lərin adını
# - hər layer-in output shape-ini
# - hər layer-dəki parametr sayını
# - ümumi parametr sayını
#
# Bu modeldə summary belə görünəcək:
#
# Conv2D          → (None, 28, 28, 32)   → 320
# MaxPooling2D    → (None, 14, 14, 32)   → 0
# Conv2D          → (None, 14, 14, 64)   → 18 496
# MaxPooling2D    → (None, 7, 7, 64)     → 0
# Flatten         → (None, 3136)         → 0
# Dense           → (None, 128)          → 401 536
# Dense           → (None, 10)           → 1 290
#
# Buradakı None → batch ölçüsüdür (şəkil sayı).
# Bu, əvvəlcədən məlum deyil, çünki model istənilən sayda şəkli qəbul edə bilər.


model.compile(optimizer='adam',
             loss='categorical_crossentropy',
             metrics=['accuracy'])
# compile() modelin necə öyrədiləcəyini müəyyən edir.
#
# Burada üç əsas parametr var:
#
# optimizer
# loss
# metrics
#
# optimizer='adam'
# → Adam optimizer-dən istifadə edirik.
# → Optimizer modelin weight və bias qiymətlərini dəyişdirərək
#   loss funksiyasını azaltmağa çalışır.
# → Adam machine learning-də ən çox istifadə olunan optimizerlərdən biridir.
#
# loss='categorical_crossentropy'
# → Classification problemi üçün loss funksiyasıdır.
# → Modelin verdiyi ehtimallarla düzgün One-Hot cavab arasındakı
#   fərqi ölçür.
# → Bunun işləməsi üçün cavablar One-Hot formada olmalıdır,
#   ona görə yuxarıda to_categorical() istifadə etmişdik.
# → Modelin məqsədi: LOSS → mümkün qədər kiçik olsun.
#
# metrics=['accuracy']
# → təlim zamanı accuracy göstəricisini də hesablamağımızı istəyirik.
# → Accuracy modelin neçə faizlik düzgün cavab verdiyini göstərir.
#
# Bu hissə əvvəlki kodla tam eynidir.
# Optimizer, loss və metrics dəyişməyib —
# dəyişən yalnız modelin daxili quruluşudur.


his = model.fit(x_train, y_train_cat, batch_size=32, epochs=5, validation_split=0.2)
# fit() modelin öyrədilməsini başladır.
#
# Əvvəlki koddan fərq:
# burada fit()-in nəticəsi his adlı dəyişənə mənimsədilir (his = history).
#
# Bu dəyişənin içində təlim tarixçəsi saxlanılır.
# Yəni hər epoch üçün loss və accuracy dəyərləri yadda qalır.
# Lazım olarsa, sonradan onları qrafik şəklində göstərmək olar, məsələn:
#
# his.history['loss']          → training loss (hər epoch üçün)
# his.history['accuracy']      → training accuracy
# his.history['val_loss']      → validation loss
# his.history['val_accuracy']  → validation accuracy
#
# Parametrlər:
#
# x_train
# → modelə verilən giriş şəkilləridir.
# → Forması artıq (60000, 28, 28, 1)-dir (Conv2D üçün uyğun).
#
# y_train_cat
# → həmin şəkillərin düzgün One-Hot cavablarıdır.
#
# batch_size=32
# → model məlumatları 32-lik qruplar şəklində emal edir.
#
# Məsələn:
#
# 32 şəkil → model → weight-lər yenilənir
# 32 şəkil → model → weight-lər yenilənir
# ...
#
# epochs=5
# → bütün training dataset-i 5 dəfə modeldən keçiriləcək.
#
# 1 epoch = bütün training datasetinin model tərəfindən
# bir dəfə görülməsi.
#
# validation_split=0.2
# → training məlumatlarının SON 20%-ni validation üçün ayırırıq.
#   (Vacib: təsadüfi yox, məhz son 20% götürülür.)
#
# Yəni təxminən:
#
# 80% (48 000 şəkil) → modelin öyrənməsi
# 20% (12 000 şəkil) → modelin təlim zamanı yoxlanılması
#
# Hər epoch-da 48 000 / 32 = 1500 batch (addım) işləyəcək.
#
# Validation məlumatları modelin öyrənmə zamanı
# yeni məlumatlar üzərində necə nəticə verdiyini izləməyə kömək edir.
# Model bu 12 000 şəkil üzərində təlim keçmir, yalnız yoxlanılır.
#
# Qeyd:
# CNN Dense şəbəkəsindən daha ağır hesablama tələb edir,
# ona görə təlim əvvəlkindən daha uzun çəkə bilər.


model.evaluate(x_test, y_test_cat)
# evaluate() modelin test datasetindəki nəticəsini ölçür.
#
# x_test
# → modelin training zamanı istifadə etmədiyi test şəkilləri.
# → Forması (10000, 28, 28, 1)-dir.
#
# y_test_cat
# → həmin şəkillərin düzgün One-Hot cavabları.
#
# Nəticədə test loss və test accuracy əldə edirik.
#
# Məsələn:
#
# loss = 0.03
# accuracy = 0.99
#
# accuracy 0.99 olarsa,
# bu təxminən 99% düzgün nəticə deməkdir.
#
# Adətən bu CNN modeli əvvəlki sadə Dense modelindən (təxminən 97%)
# daha yüksək dəqiqlik verir (təxminən 99%),
# çünki Conv2D qatları şəkilin məkan quruluşunu (xətlərin,
# əyrilərin yerini) nəzərə alır. Flatten ilə birbaşa Dense-ə verəndə
# isə bu məkan məlumatı itirilirdi.
#
# Qeyd:
# Bu kodda təkbaşına model.evaluate() yazılıb, nəticə heç bir dəyişənə
# mənimsədilmir. Ona görə yalnız konsolda çap olunur.